### Assignement 2 
### MLOPS
### Assignment: API Cost, Latency and Prompt Version Management
### Student name: Meghana K U
### BITS ID : 2025AE05640



### Installing Required Libraries
This cell installs all required Python libraries used throughout the assignment. These libraries support local LLM execution, experiment tracking, token analysis, data processing, and visualization.

Libraries used:

ollama
pandas
numpy
mlflow
matplotlib
seaborn
tiktoken

In [1]:
!pip install openai pandas numpy python-dotenv mlflow

In [2]:
!pip install pandas
!pip install numpy
!pip install python-dotenv
!pip install mlflow
!pip install matplotlib
!pip install seaborn
!pip install jupyter
!pip install ipykernel

#### Import Required Packages
This cell imports all required Python modules used throughout the experiment. These libraries are responsible for data manipulation, latency measurement, token analysis, MLflow tracking, and communication with the Phi-3 local language model.

In [3]:
import openai
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

print("All libraries installed successfully!")

All libraries installed successfully!


In [4]:
!pip install ollama pandas numpy mlflow matplotlib seaborn

In [5]:
from ollama import chat

response = chat(
    model='phi3',
    messages=[
        {
            'role': 'user',
            'content': 'What is Machine Learning?'
        }
    ]
)

print(response['message']['content'])


Machine Learning is a subfield of artificial intelligence (AI) focused on developing algorithms that allow computers to learn from and make predictions or decisions based on data. This involves creating models that can identify patterns and make inferences, often without being explicitly programmed to do so. Machine learning algorithms improve their performance through experience, typically through a process called training, where the system is fed large amounts of data and continuously adjusted to improve accuracy in tasks such as prediction, classification, and regression.


There are various types of machine learning algorithms, each suited to different tasks and data types. Supervised learning involves labeled data and is used for tasks like image recognition and spam detection, where the correct output is known. Unsupervised learning deals with unlabeled data and is often used for clustering and association problems, such as market basket analysis. Reinforcement learning involves 

In [6]:
import time
import json
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

from ollama import chat

print("Environment Ready")

Environment Ready


#### Define Evaluation Questions
This cell creates a list of five academic questions that will be used consistently throughout the experiment. Using the same set of questions for all prompt versions ensures a fair comparison of latency, token usage, cost, and response quality.

In [7]:
questions = [
    "What is Machine Learning?",
    "Explain Supervised Learning.",
    "What is Overfitting?",
    "What is Gradient Descent?",
    "Difference between Classification and Regression?"
]

len(questions)

5

#### Prompt Version 1 (Basic Prompt)
This prompt represents the baseline version used for evaluation. The prompt simply instructs the model to answer a student question clearly without any additional structure or formatting requirements.

In [8]:
prompt_v1 = """
Answer the following student question clearly.

Question: {question}
"""

### Measure Response Latency
This cell sends each academic question to the Phi-3 model and records:

Start time
End time
Response latency
Generated answer
The collected data is stored in a DataFrame for further analysis.

#### Create Results DataFrame
This cell converts the collected response information into a Pandas DataFrame. The DataFrame serves as the central dataset for latency analysis, token counting, and cost calculation.

In [9]:
results = []

for idx, question in enumerate(questions):

    start_time = time.time()

    response = chat(
        model='phi3',
        messages=[
            {
                'role': 'user',
                'content': prompt_v1.format(
                    question=question
                )
            }
        ]
    )

    end_time = time.time()

    latency = end_time - start_time

    answer = response['message']['content']

    results.append({
    'question_id': idx+1,
    'prompt_version': 'v1',
    'model_name': 'phi3',
    'start_time': start_time,
    'end_time': end_time,
    'latency_seconds': latency,
    'status': 'success',
    'question': question,
    'response': answer
})

df = pd.DataFrame(results)

df

,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status,question,response
0,1,v1,phi3,1.793896e+09,1.793896e+09,14.562004,success,What is Machine Learning?,\nMachine Learning is a subset of artificial i...
1,2,v1,phi3,1.793896e+09,1.793896e+09,24.139617,success,Explain Supervised Learning.,\nSupervised Learning is a type of machine lea...
2,3,v1,phi3,1.793896e+09,1.793896e+09,35.727879,success,What is Overfitting?,\nOverfitting is a common problem in machine l...
3,4,v1,phi3,1.793896e+09,1.793896e+09,16.974736,success,What is Gradient Descent?,\nGradient Descent is an optimization algorith...
4,5,v1,phi3,1.793896e+09,1.793896e+09,35.737387,success,Difference between Classification and Regression?,\nClassification and regression are two types ...


#### Calculate Latency Statistics
This cell computes key latency metrics including:

Mean latency
Median latency
Minimum latency
Maximum latency
Standard deviation
These metrics help evaluate the responsiveness and performance consistency of the model.

In [10]:
print("Mean:", df["latency_seconds"].mean())
print("Median:", df["latency_seconds"].median())
print("Min:", df["latency_seconds"].min())
print("Max:", df["latency_seconds"].max())
print("Std:", df["latency_seconds"].std())

Mean: 25.42832455635071
Median: 24.13961696624756
Min: 14.56200385093689
Max: 35.73738694190979
Std: 10.044381824219862


#### Export Latency Results
This cell saves all latency measurements and response information to a CSV file named:

latency_tokens_cost.csv

The CSV file is included as part of the assignment deliverables.

In [11]:
df.to_csv(
    "latency_tokens_cost.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [12]:
!pip install tiktoken

#### Initialize Token Counter
This cell initializes the tokenizer used to estimate token usage. Token counts are required for cost calculation and efficiency analysis.

In [13]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(str(text)))

#### Create Token Counting Function
This function accepts input text and returns the total number of tokens generated by the tokenizer. It is used to calculate input, output, and total token consumption.

In [14]:
df["input_tokens"] = df["question"].apply(
    count_tokens
)

df["output_tokens"] = df["response"].apply(
    count_tokens
)

df["total_tokens"] = (
    df["input_tokens"] +
    df["output_tokens"]
)

df[
    [
        "question_id",
        "input_tokens",
        "output_tokens",
        "total_tokens"
    ]
]

,question_id,input_tokens,output_tokens,total_tokens
0,1,5,99,104
1,2,6,172,178
2,3,6,239,245
3,4,6,123,129
4,5,6,216,222


#### Calculate Token Usage
This cell calculates:

Input tokens

Output tokens

Total tokens
for each question-response pair. These values are later used for cost estimation.

In [15]:
INPUT_PRICE = 0.15
OUTPUT_PRICE = 0.60

#### Calculate Cost Per Request
This cell estimates:

Input token cost

Output token cost

Total request cost

using the calculated token counts and predefined pricing information.

In [16]:
df["input_cost"] = (
    df["input_tokens"] / 1_000_000
) * INPUT_PRICE

df["output_cost"] = (
    df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE

df["total_cost"] = (
    df["input_cost"] +
    df["output_cost"]
)

df[
    [
        "question_id",
        "input_tokens",
        "output_tokens",
        "total_cost"
    ]
]

,question_id,input_tokens,output_tokens,total_cost
0,1,5,99,0.000060
1,2,6,172,0.000104
2,3,6,239,0.000144
3,4,6,123,0.000075
4,5,6,216,0.000130


#### Define Pricing Model
This cell defines token pricing assumptions based on GPT-4o-mini pricing rates. Since the project uses a local Phi-3 model, these rates are used only for educational cost estimation purposes.

#### Monthly Cost Projection
This cell calculates the projected operational cost for 10,000 requests based on the average cost per request. This provides insight into large-scale deployment expenses.

In [17]:
avg_cost = df["total_cost"].mean()

monthly_cost = avg_cost * 10000

print(f"Average Cost Per Request: ${avg_cost:.8f}")
print(f"Projected Monthly Cost for 10,000 Requests: ${monthly_cost:.4f}")

Average Cost Per Request: $0.00010275
Projected Monthly Cost for 10,000 Requests: $1.0275


#### Create Prompt Versions
This section defines three prompt versions:

Prompt V1 (Basic)

Prompt V2 (Academic Assistant)

Prompt V3 (Structured JSON Output
          
These prompt versions will be evaluated and compared using the same set of academic questions.

In [18]:
prompt_v1 = """
Answer the following student question clearly.

Question: {question}
"""


In [19]:
prompt_v2 = """
You are an academic assistant.

Answer the following student question in a clear,
concise and accurate manner.

Use simple language suitable for undergraduate students.

Question: {question}
"""

In [20]:
prompt_v3 = """
You are an academic assistant.

Answer the student question and return ONLY valid JSON.

Do not include markdown or explanations.

{
    "answer":"string",
    "confidence":"high|medium|low",
    "key_points":["string"]
}

Question: {question}
"""

#### Create MLflow Experiment
This cell creates and initializes the MLflow experiment:

eduassist_llmops_assignment

The experiment is used to track prompt performance and evaluation metrics.

In [21]:
import mlflow

mlflow.set_experiment(
    "eduassist_llmops_assignment"
)

print("Experiment Created")

Experiment Created


In [22]:
import json
import time
import pandas as pd
import mlflow
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(str(text)))

def is_valid_json(text):
    try:
        obj = json.loads(text)

        required_keys = [
            "answer",
            "confidence",
            "key_points"
        ]

        return int(
            all(
                key in obj
                for key in required_keys
            )
        )

    except:
        return 0

# FIXED PROMPTS

prompt_v1 = """
Answer the following student question clearly.

Question: {question}
"""

prompt_v2 = """
You are an academic assistant.

Answer the following student question in a clear,
concise and accurate manner.

Use simple language suitable for undergraduate students.

Question: {question}
"""

prompt_v3 = """
You are an academic assistant.

Answer the student question and return ONLY valid JSON.

Do not include markdown or explanations.

{{
    "answer":"string",
    "confidence":"high|medium|low",
    "key_points":["string"]
}}

Question: {question}
"""

prompts = {
    "v1": prompt_v1,
    "v2": prompt_v2,
    "v3": prompt_v3
}

evaluation_results = []

for version, prompt in prompts.items():

    latencies = []
    input_tokens_list = []
    output_tokens_list = []
    costs = []
    json_scores = []

    print(f"Running {version}...")

    for question in questions:

        start_time = time.time()

        response = chat(
            model="phi3",
            messages=[
                {
                    "role": "user",
                    "content": prompt.format(
                        question=question
                    )
                }
            ]
        )

        end_time = time.time()

        answer = response["message"]["content"]

        latency = end_time - start_time

        input_tokens = count_tokens(question)
        output_tokens = count_tokens(answer)

        cost = (
            (input_tokens / 1_000_000) * INPUT_PRICE +
            (output_tokens / 1_000_000) * OUTPUT_PRICE
        )

        json_score = (
            is_valid_json(answer)
            if version == "v3"
            else 0
        )

        latencies.append(latency)
        input_tokens_list.append(input_tokens)
        output_tokens_list.append(output_tokens)
        costs.append(cost)
        json_scores.append(json_score)

    avg_latency = sum(latencies) / len(latencies)

    avg_input_tokens = (
        sum(input_tokens_list) /
        len(input_tokens_list)
    )

    avg_output_tokens = (
        sum(output_tokens_list) /
        len(output_tokens_list)
    )

    avg_cost = (
        sum(costs) /
        len(costs)
    )

    valid_json_rate = (
        sum(json_scores) /
        len(json_scores)
    )

    with mlflow.start_run(
        run_name=version
    ):

        mlflow.log_metric(
            "avg_latency_seconds",
            avg_latency
        )

        mlflow.log_metric(
            "avg_input_tokens",
            avg_input_tokens
        )

        mlflow.log_metric(
            "avg_output_tokens",
            avg_output_tokens
        )

        mlflow.log_metric(
            "avg_cost_per_request",
            avg_cost
        )

        mlflow.log_metric(
            "valid_json",
            valid_json_rate
        )

        mlflow.set_tag(
            "prompt_version",
            version
        )

    evaluation_results.append({
        "Prompt Version": version,
        "Avg Latency": round(avg_latency, 2),
        "Avg Input Tokens": round(avg_input_tokens, 2),
        "Avg Output Tokens": round(avg_output_tokens, 2),
        "Avg Cost": round(avg_cost, 8),
        "Valid JSON Rate": round(valid_json_rate, 2)
    })

comparison_df = pd.DataFrame(
    evaluation_results
)

comparison_df

Running v1...
Running v2...
Running v3...


,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate
0,v1,38.27,5.8,243.4,0.000147,0.0
1,v2,21.73,5.8,145.0,0.000088,0.0
2,v3,17.87,5.8,97.4,0.000059,0.0


#### JSON Validation Function
This function verifies whether a response follows the expected JSON schema. The validation result is used to calculate the valid_json metric for Prompt Version 3.

In [23]:
comparison_df.to_csv(
    "prompt_evaluation_results.csv",
    index=False
)

comparison_df

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate
0,v1,38.27,5.8,243.4,0.000147,0.0
1,v2,21.73,5.8,145.0,0.000088,0.0
2,v3,17.87,5.8,97.4,0.000059,0.0


In [24]:
comparison_df.to_csv(
    "prompt_evaluation_results.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [25]:
!mlflow ui

^C


#### Evaluate Prompt Versions
This cell executes all three prompt versions against the same set of academic questions.

For each prompt version, the following metrics are collected:

Average latency
Average input tokens
Average output tokens
Average cost per request
JSON validity score
The collected results are stored for comparison.

In [28]:
import mlflow

runs = mlflow.search_runs(
    experiment_names=["eduassist_llmops_assignment"]
)

runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.avg_cost_per_request,metrics.avg_output_tokens,metrics.avg_input_tokens,metrics.valid_json,metrics.avg_latency_seconds,tags.mlflow.source.name,tags.mlflow.source.type,tags.prompt_version,tags.mlflow.runName,tags.mlflow.user
0,ff8fbff318ca4d5e9690fea503af7b65,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 16:32:33.297000+00:00,2026-11-05 16:32:33.663000+00:00,0.000059,97.4,5.8,0.0,17.874323,mops_assignment2.ipynb,NOTEBOOK,v3,v3,Meghana
1,2ad58aadb53d43858eb9eb39ed15e1ca,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 16:31:02.753000+00:00,2026-11-05 16:31:03.455000+00:00,0.000088,145.0,5.8,0.0,21.728264,mops_assignment2.ipynb,NOTEBOOK,v2,v2,Meghana
2,9b8491fafecc4fd6890562ae40dab103,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 16:29:13.177000+00:00,2026-11-05 16:29:13.682000+00:00,0.000147,243.4,5.8,0.0,38.267074,mops_assignment2.ipynb,NOTEBOOK,v1,v1,Meghana
3,6f2e98a3aaf94b80aeaa47ee83769654,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 16:04:05.219000+00:00,2026-11-05 16:04:05.531000+00:00,0.000095,156.2,5.8,0.0,24.422866,mops_assignment2.ipynb,NOTEBOOK,v2,v2,Meghana
4,5b3d3bae2a5d4417a83529e938224071,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 16:02:02.346000+00:00,2026-11-05 16:02:02.874000+00:00,0.000131,216.6,5.8,0.0,33.429996,mops_assignment2.ipynb,NOTEBOOK,v1,v1,Meghana
5,3e11801bc460434db16d065e47eb19b5,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 14:09:37.191000+00:00,2026-11-05 14:09:37.448000+00:00,0.000175,289.8,5.8,0.0,63.889475,mops_assignment2.ipynb,NOTEBOOK,v3,v3,Meghana
6,0250fc510b734c4692532df83a28156d,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 14:04:17.130000+00:00,2026-11-05 14:04:17.350000+00:00,0.000082,135.8,5.8,0.0,20.476060,mops_assignment2.ipynb,NOTEBOOK,v2,v2,Meghana
7,4f680541d1bd45c28c2c43631da09f5b,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 14:02:34.394000+00:00,2026-11-05 14:02:34.660000+00:00,0.000097,160.2,5.8,0.0,22.903323,mops_assignment2.ipynb,NOTEBOOK,v1,v1,Meghana
8,db45009c1f624304a2df5ab3e1cb9ba3,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 09:04:37.170000+00:00,2026-11-05 09:04:37.575000+00:00,0.000086,142.0,5.8,0.0,23.795890,mops_assignment2.ipynb,NOTEBOOK,v3,v3,Meghana
9,b436dd11533143b9809d07d2af0c1fa3,1,FINISHED,file:C:/Users/Meghana/Downloads/Assignment2_ML...,2026-11-05 09:02:37.633000+00:00,2026-11-05 09:02:37.968000+00:00,0.000078,127.8,5.8,0.0,20.784761,mops_assignment2.ipynb,NOTEBOOK,v2,v2,Meghana


#### Log Metrics to MLflow
This cell logs all evaluation metrics into MLflow. Each prompt version is recorded as an independent run to enable experiment tracking and comparison.

In [29]:
runs[[
    "tags.mlflow.runName",
    "metrics.avg_latency_seconds",
    "metrics.avg_input_tokens",
    "metrics.avg_output_tokens",
    "metrics.avg_cost_per_request",
    "metrics.valid_json"
]]

,tags.mlflow.runName,metrics.avg_latency_seconds,metrics.avg_input_tokens,metrics.avg_output_tokens,metrics.avg_cost_per_request,metrics.valid_json
0,v3,17.874323,5.8,97.4,0.000059,0.0
1,v2,21.728264,5.8,145.0,0.000088,0.0
2,v1,38.267074,5.8,243.4,0.000147,0.0
3,v2,24.422866,5.8,156.2,0.000095,0.0
4,v1,33.429996,5.8,216.6,0.000131,0.0
5,v3,63.889475,5.8,289.8,0.000175,0.0
6,v2,20.476060,5.8,135.8,0.000082,0.0
7,v1,22.903323,5.8,160.2,0.000097,0.0
8,v3,23.795890,5.8,142.0,0.000086,0.0
9,v2,20.784761,5.8,127.8,0.000078,0.0
